In this notebook we are trying to write the code to perform multiple streaming jobs. Its same as working in medalian arch, one job to load the data to raw and another job to load the data to silver.

In [0]:
class bronze():
    def __init__(self):
        self.directory_path = '/Volumes/workspace/word_count/sample/invoice_source/'
        # we have to use the base location path, cannot directly point to the exact file
    
    def schema(self):
        return( """InvoiceNumber string, CreatedTime bigint, StoreID string, PosID string, CashierID string,
                CustomerType string, CustomerCardNo string, TotalAmount double, NumberOfItems bigint, 
                PaymentMethod string, TaxableAmount double, CGST double, SGST double, CESS double, 
                DeliveryType string,
                DeliveryAddress struct<AddressLine string, City string, ContactNumber string, PinCode string, 
                State string>,
                InvoiceLineItems array<struct<ItemCode string, ItemDescription string, 
                    ItemPrice double, ItemQty bigint, TotalValue double>>
            """
        # here there are two columns which are hvaing the copmlex structure, one is struct and another is array of struct. So the same has to be declared while declaring the schema.
        #have to check also how it works without using the schema as well
        )

    def read_invoice(self):
        return(spark.readStream
               .format('json')
               .schema(self.schema())
               .option('maxFilesPerTrigger', 1) # this is used when the file size is too high else 1 file per trigger is very small. By default it is 1000 files per trigger.
               .option('multiLine', True)
               .option('cleanSource','delete') # this will delete all the consumed files but as little harsh to delete we have another option to archive
                .option('sourceArchiveDir','/Volumes/workspace/word_count/sample/invoice/archive/') # this will just archive the files
               .load(self.directory_path)
        )
    # For the batch and stream the read will be same.

    # def explode_invoice(self,invoice_df):
    #     return(
    #         invoice_df.selectExpr("InvoiceNumber", "CreatedTime", "StoreID", "PosID",
    #                                   "CustomerType", "PaymentMethod", "DeliveryType", "DeliveryAddress.City",
    #                                   "DeliveryAddress.State","DeliveryAddress.PinCode", 
    #                                   "explode(InvoiceLineItems) as LineItem"
    #                             )
              # here using the select Expr is used when we want to apply sql type of operations like case stements or alias on each column or explode etc. while just the select function will be used majorly, but if the transformation on column is high and needs to copy direclty from the sql syntax we can use the selectExpr
        #)

    # def flatten_invoice(self,explode_df):
    #     from pyspark.sql.functions import expr
    #     return(
    #         explode_df.withColumn('item',expr('LineItem.ItemCode')).
    #                                     withColumn('item_description',expr('LineItem.ItemDescription')).
    #                                     withColumn('item_price',expr('LineItem.ItemPrice')).
    #                                     withColumn('item_qty',expr('LineItem.ItemQty')).
    #                                     drop('LineItem')
    #  # here the with column is used to make the exploded column into new column. drop the complex Lineitem column. We have to use the fucntion expr and have to import to use it.
    #     )

    def write_df(self,flatten_df, trigger='once'):
        # but for the write operation there will be small change in the trigger config.
        sQuery=flatten_df.writeStream.format('delta').option('checkpointLocation','/Volumes/workspace/word_count/sample/invoice/checkpoint_bronze/').outputMode('append')

        if trigger=='once':
            sQuery=sQuery.trigger(once=True)
        else:
            sQuery=sQuery.trigger(processingTime=trigger)
        sQuery=sQuery.toTable('invoice_data_bronze')
        return(sQuery)

    def process(self,trigger):
        print('Starting the stream job---', end='')
        invoice_schema=self.schema()
        invoice_df=self.read_invoice()
        #explode_df=self.explode_invoice(invoice_df)
        #flatten_df=self.flatten_invoice(explode_df)
        write_df=self.write_df(invoice_df,trigger)
        print('done')
        return(write_df)

In [0]:
bronze_obj=bronze()
write_df=bronze_obj.process('once')
display(spark.table('invoice_data_bronze'))

In [0]:
class silver():
    def __init__(self):
        self.directory_path = '/Volumes/workspace/word_count/sample/invoice_source'
        # we have to use the base location path, cannot directly point to the exact file


    def read_invoice(self):
        return(spark.readStream.table('invoice_data_bronze')
        )
    # For the batch and stream the read will be same.

    def explode_invoice(self,invoice_df):
        return(
            invoice_df.selectExpr("InvoiceNumber", "CreatedTime", "StoreID", "PosID",
                                      "CustomerType", "PaymentMethod", "DeliveryType", "DeliveryAddress.City",
                                      "DeliveryAddress.State","DeliveryAddress.PinCode", 
                                      "explode(InvoiceLineItems) as LineItem"
                                )
              # here using the select Expr is used when we want to apply sql type of operations like case stements or alias on each column or explode etc. while just the select function will be used majorly, but if the transformation on column is high and needs to copy direclty from the sql syntax we can use the selectExpr
        )

    def flatten_invoice(self,explode_df):
        from pyspark.sql.functions import expr
        return(
            explode_df.withColumn('item',expr('LineItem.ItemCode')).
                                        withColumn('item_description',expr('LineItem.ItemDescription')).
                                        withColumn('item_price',expr('LineItem.ItemPrice')).
                                        withColumn('item_qty',expr('LineItem.ItemQty')).
                                        drop('LineItem')
     # here the with column is used to make the exploded column into new column. drop the complex Lineitem column. We have to use the fucntion expr and have to import to use it.
        )

    def write_df(self,flatten_df, trigger='once'):
        # but for the write operation there will be small change in the trigger config.
        sQuery=flatten_df.writeStream.format('delta').option('checkpointLocation','/Volumes/workspace/word_count/sample/invoice/checkpoint/invoice_data_silver/').outputMode('append')
        #trigger(once=True). # we can use 'processingTime' and '2 seconds' for every 2 sec run
        
        #toTable('invoice_data_1')
        
        if trigger=='once':
            sQuery=sQuery.trigger(once=True)
        else:
            sQuery=sQuery.trigger(processingTime=trigger)
        sQuery=sQuery.toTable('invoice_data_silver')
        return(sQuery)

    def process(self,trigger):
        print('Starting the stream job---', end='')
        invoice_df=self.read_invoice()
        explode_df=self.explode_invoice(invoice_df)
        flatten_df=self.flatten_invoice(explode_df)
        write_df=self.write_df(flatten_df,trigger)
        print('done')
        return(write_df)

In [0]:
silver_obj=silver()
silver_obj.process('once')
display(spark.table('invoice_data_silver'))


in the above code we have done both the ingestion to bronze and then creating the silver. Looks like in streaming application we are creating the delta table in bonze layer itself which is same in the DHL
